In [2]:
# Uncomment in case packages are not installed
!pip install timm
!pip install transformers
!pip install pyspellchecker
!pip install torch torchvision
!pip install ftfy
!pip install regex
!pip install sentencepiece

In [4]:
import torch
from torch import nn
import argparse
import numpy as np

torch.cuda.empty_cache()
import pandas as pd
import re

import os
import cv2
import gc
import itertools
from tqdm.autonotebook import tqdm
import albumentations as A

import torch.nn.functional as F
import timm


from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
#For displaying loadbar
from tqdm import tqdm

from collections import Counter

# from spellchecker import SpellChecker

import spacy
import operator

# This is our visual library
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
import math
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.metrics import confusion_matrix

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to /usr/share/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package names to /usr/share/nltk_data...
[nltk_data]   Package names is already up-to-date!


/tmp/ipykernel_25/3378561779.py:50: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use('seaborn-whitegrid')


# Initialization

In [5]:
class CFG:
    # If there's a GPU available...
    if torch.cuda.is_available():
      # Tell PyTorch to use the GPU.
      device = torch.device("cuda")
      print('There are %d GPU(s) available.' % torch.cuda.device_count())
      print('We will use the GPU:', torch.cuda.get_device_name())
    # If not...
    else:
      print('No GPU available, using the CPU instead.')
      device = torch.device("cpu")

    image_path = "/kaggle/input/exist-memes-dataset/EXIST 2024 Memes Dataset/EXIST 2024 Memes Dataset/training/memes/"
    train_file = '/kaggle/input/exist-dataset-folds/fold_0_train.csv'
    test_file = '/kaggle/input/exist-dataset-folds/fold_0_val.csv'




    # for projection head; used for both image and text encoders
    num_projection_layers = 1
    projection_dim = 256
    dropout = 0.1
    # dropout = 0.4


There are 1 GPU(s) available.
We will use the GPU: Tesla P100-PCIE-16GB


In [7]:
# Count the number of samples with 0 and 1 in the 'label' column
df_train = pd.read_csv(CFG.train_file, header='infer', keep_default_na=False)
df_train = df_train.sample(frac=1).reset_index(drop=True)

# Drop rows where 'value' is either NaN or an empty string
df_train = df_train[~df_train['value'].isin(['', None])]
df_train = df_train.reset_index(drop=True)

# Verify the change
print(df_train['value'].value_counts())

df_train['value'] = df_train['value'].map({'YES': 1, 'NO': 0}).fillna(df_train['value']).astype(int)

df_val = pd.read_csv(CFG.test_file, header='infer', keep_default_na=False)

# Drop rows where 'value' is either NaN or an empty string
df_val = df_val[~df_val['value'].isin(['', None])]

# Reset the index after dropping rows
df_val = df_val.reset_index(drop=True)

df_val['value'] = df_val['value'].map({'YES': 1, 'NO': 0}).fillna(df_val['value']).astype(int)

# Check the updated dataframe

print(df_val['value'].value_counts())

train_image_names = df_train['meme']
train_image_path = []
for name in train_image_names:
  train_image_path.append(CFG.image_path + name)

train_texts =  df_train['text']
train_texts = train_texts.astype(str)
train_labels =  df_train['value']

val_image_names = df_val['meme']
val_image_path = []
for name in val_image_names:
  val_image_path.append(CFG.image_path + name)

val_texts =  df_val['text']
val_texts = val_texts.astype(str)
test_labels =  df_val['value']


value
YES    1833
NO     1243
Name: count, dtype: int64
value
1    205
0    139
Name: count, dtype: int64


# CLIP Model

In [8]:
from transformers import CLIPProcessor, CLIPModel


# Load CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.to(CFG.device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [9]:
class MemeDataset(Dataset):
    def __init__(self, image_paths, texts, labels, processor, max_length=77):
        self.image_paths = image_paths
        self.texts = texts
        self.labels = labels
        self.processor = processor
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Process text and image
        inputs = self.processor(text=text, images=image, return_tensors="pt", padding="max_length", max_length=self.max_length, truncation=True)
        inputs["label"] = torch.tensor(label, dtype=torch.long)
        return inputs
    

In [10]:
class CLIPClassificationHead(nn.Module):
    def __init__(self, clip_model, hidden_dim=512):
        super(CLIPClassificationHead, self).__init__()
        self.clip_model = clip_model
        self.fc = nn.Linear(clip_model.config.projection_dim, 2)  # Binary classification: 2 classes
        # self.fc = nn.Linear(clip_model.config.projection_dim * 2, 2)

    def forward(self, input_ids, pixel_values, attention_mask=None):
        outputs = self.clip_model(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask)
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds
        
        # # Combine the embeddings (e.g., concatenate, average, or sum)
        combined_features = (image_embeds + text_embeds) / 2

        # combined_features = torch.cat([image_embeds, text_embeds], dim=-1)
        
        # Pass through classifier head
        logits = self.fc(combined_features)
        return logits

In [11]:
def train(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].squeeze(1).to(device)
        pixel_values = batch["pixel_values"].squeeze(1).to(device)
        labels = batch["label"].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        logits = model(input_ids=input_ids, pixel_values=pixel_values)
        
        # Calculate loss and backpropagate
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [12]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

# Initialize model, loss function, optimizer
clip_with_head = CLIPClassificationHead(model).to(CFG.device)
loss_fn = CrossEntropyLoss()
optimizer = AdamW(clip_with_head.parameters(), lr=1e-5)

In [13]:
from torch.utils.data import DataLoader
import time
# Define the testing function
start = time.time()
print("Start time:",start)

# Assuming you have a pandas dataframe `train_df` and `val_df`
train_dataset = MemeDataset(train_image_path, train_texts, train_labels, processor)
# val_dataset = MemeDataset(val_df, processor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=8)

epochs = 10
for epoch in range(epochs):
    train_loss = train(clip_with_head, train_loader, optimizer, loss_fn, CFG.device)
#     val_accuracy = evaluate(clip_with_head, val_loader, device)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}")

end = time.time()
runtime_seconds = end - start
print(f"Execution time: {runtime_seconds:.2f} seconds")
print(f"Execution time: {runtime_seconds / 60:.2f} minutes")

Start time: 1768924789.6358802
Epoch 1/10, Train Loss: 0.6494
Epoch 2/10, Train Loss: 0.6201
Epoch 3/10, Train Loss: 0.5950
Epoch 4/10, Train Loss: 0.5795
Epoch 5/10, Train Loss: 0.5722
Epoch 6/10, Train Loss: 0.5461
Epoch 7/10, Train Loss: 0.5321
Epoch 8/10, Train Loss: 0.5189
Epoch 9/10, Train Loss: 0.4871
Epoch 10/10, Train Loss: 0.4667
Execution time: 1034.91 seconds
Execution time: 17.25 minutes


In [14]:
start = time.time()
print("Start time:",start)

# Define the testing function
def test(model, dataloader, device):
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].squeeze(1).to(device)
            pixel_values = batch["pixel_values"].squeeze(1).to(device)
            labels = batch["label"].to(device)
            
            logits = model(input_ids=input_ids, pixel_values=pixel_values)
            predictions = torch.argmax(logits, dim=1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_predictions, all_labels


# Create the test DataLoader
test_dataset = MemeDataset(val_image_path, val_texts, test_labels, processor)
test_loader = DataLoader(test_dataset, batch_size=8)

all_predictions, all_labels = test(clip_with_head, test_loader, CFG.device)

end = time.time()
runtime_seconds = end - start
print(f"Execution time: {runtime_seconds:.2f} seconds")
print(f"Execution time: {runtime_seconds / 60:.2f} minutes")

Start time: 1768925839.6589224
Execution time: 15.73 seconds
Execution time: 0.26 minutes


In [15]:
from sklearn.metrics import roc_auc_score, f1_score

# Count the number of correct 1 labels
correct_1_labels = sum(pred == orig == 1 for pred, orig in zip(all_predictions, test_labels))

# Count the number of correct 0 labels
correct_0_labels = sum(pred == orig == 0 for pred, orig in zip(all_predictions, test_labels))

# Print the results
print("Number of correct 1 labels:", correct_1_labels)
print("Number of correct 0 labels:", correct_0_labels)

target_names = ['0', '1']
print(classification_report(test_labels, all_predictions, target_names=target_names), '\n')

cm = confusion_matrix(test_labels, all_predictions)
cm_df = pd.DataFrame(cm, index=['True_0', 'True_1'], columns=['Pred_0', 'Pred_1'])

# Print the confusion matrix
print("Confusion Matrix:")
print(cm_df)
    
auc = roc_auc_score(test_labels, all_predictions)
print("Area under curve:", auc )
f1s = f1_score(all_labels, all_predictions)
print(f1s)

print("Predicted labels:", all_predictions)
# Convert to a DataFrame
df = pd.DataFrame({"Predicted Labels": all_predictions})

# Save to CSV
df.to_csv("/kaggle/working/EXIST_misogynous_CLIP_10_fold4.csv", index=False)

Number of correct 1 labels: 146
Number of correct 0 labels: 88
              precision    recall  f1-score   support

           0       0.60      0.63      0.62       139
           1       0.74      0.71      0.73       205

    accuracy                           0.68       344
   macro avg       0.67      0.67      0.67       344
weighted avg       0.68      0.68      0.68       344
 

Confusion Matrix:
        Pred_0  Pred_1
True_0      88      51
True_1      59     146
Area under curve: 0.6726443235655379
0.7263681592039801
Predicted labels: [1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1,